# Fracture: legacy image-to-curvature analysis

This cleaned notebook retains the polynomial/spline workflow in the supplied `break.ipynb`.
The author confirmed that this algorithm produced the reported fracture results. The plastic physics-informed approach is an optional refinement requiring separate validation.
Read [README.md](README.md) for the distinction and the separate fracture-property helper.
Select the final intact frame immediately before separation; the code does not select it automatically.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

HERE = Path.cwd() if (Path.cwd() / "curvature.py").is_file() else Path.cwd() / "fracture"
if not (HERE / "curvature.py").is_file():
    raise RuntimeError("Start Jupyter in the repository root or the fracture folder")
sys.path.insert(0, str(HERE.resolve()))
from curvature import analyze_image

## 1. Configure the image and geometry

The original image convention flips the image vertically so that the root occupies the
first detected rows. Verify this orientation in the overlay. Beam length is the physical
arc length of the entire tracked segment. The values below are examples from the source,
not universal settings for the manuscript's specimens.

In [ ]:
IMAGE = HERE / "data" / "final_intact.png"
OUTPUT = HERE / "results" / "sample_01"
BEAM_LENGTH_M = 0.075
FLIP_VERTICAL = True
HSV_LOWER = (10, 60, 30)
HSV_UPPER = (35, 255, 255)
SPLINE_SMOOTHING = 1e-4
INCLUDE_LEGACY_ADJUSTMENT = True

## 2. Extract and inspect curvature

The optional legacy heuristic clips certain curvature increases after sample 150. It is
explicitly enabled below to retain the source workflow and saved in a separate column. The raw curve is always
retained. Do not interpret this heuristic as a physical equilibrium constraint.

In [ ]:
profile = analyze_image(IMAGE, OUTPUT, beam_length_m=BEAM_LENGTH_M,
    flip_vertical=FLIP_VERTICAL, hsv_lower=HSV_LOWER, hsv_upper=HSV_UPPER,
    spline_smoothing=SPLINE_SMOOTHING,
    include_legacy_adjustment=INCLUDE_LEGACY_ADJUSTMENT)
display(Image(filename=str(OUTPUT / "diagnostics.png")))
profile.head()

## 3. Convert a verified fracture curvature to strain

Use `fracture_from_curvature.m` for `epsilon_f = max(kappa_f) * thickness / 2`.
For strength, supply the independently fitted constitutive model and its calibrated strain range.
Only a curvature field corresponding to the final intact state is appropriate. See the README
for column conventions and the current manuscript-reproduction limitation.